# Arabic Conversational AI: Standard vs Dynamic Transformer Comparison

This notebook compares a **Standard Transformer** with a **Dynamic Neural Network-Enhanced Transformer** for Arabic conversational AI.

## Goals
1. Generate ~5,000 Arabic conversational samples (greetings, Q&A, basic tasks)
2. Train a standard pure-Python transformer as baseline
3. Train a Dynamic NN transformer with adaptive architecture
4. Compare performance, efficiency, and architecture evolution

## Table of Contents
1. [Setup and Imports](#1-setup)
2. [Data Generation](#2-data)
3. [Data Exploration](#3-explore)
4. [Tokenization](#4-tokenize)
5. [Standard Transformer](#5-standard)
6. [Dynamic Transformer](#6-dynamic)
7. [Comparison](#7-comparison)
8. [Architecture Evolution](#8-architecture)
9. [Health Analysis](#9-health)
10. [Demo](#10-demo)
11. [Conclusions](#11-conclusions)

## 1. Setup and Imports <a id='1-setup'></a>

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Add parent directory to path for imports
notebook_dir = Path(os.getcwd()).parent if 'data' in os.getcwd() else Path(os.getcwd())
sys.path.insert(0, str(notebook_dir))

# Set random seed for reproducibility
SEED = 42
np.random.seed(SEED)

# Configure matplotlib for Arabic text
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print(f"NumPy version: {np.__version__}")
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Import our modules (with reload to pick up any changes)
import importlib
import src.arabic_data_generator
import src.arabic_tokenizer
import src.standard_transformer
import src.dynamic_transformer
import src.comparison_evaluator

# Reload all modules to ensure latest code
importlib.reload(src.arabic_data_generator)
importlib.reload(src.arabic_tokenizer)
importlib.reload(src.standard_transformer)
importlib.reload(src.dynamic_transformer)
importlib.reload(src.comparison_evaluator)

from src.arabic_data_generator import ArabicConversationalGenerator, generate_dataset
from src.arabic_tokenizer import ArabicTokenizer, TokenizerConfig, create_tokenizer_from_samples
from src.standard_transformer import StandardTransformer, TransformerConfig
from src.dynamic_transformer import DynamicTransformer, DynamicTransformerConfig
from src.comparison_evaluator import TransformerComparisonEvaluator, ComparisonMetrics

print("All modules imported and reloaded successfully!")

## 2. Arabic Conversational Data Generation <a id='2-data'></a>

Generate ~5,000 Arabic conversational samples across 8 categories:
- Greetings
- Personal Information
- Time/Date
- Location
- Weather
- Calculations
- General Information
- Code-switching (Arabic-English mixed)

In [ ]:
# Generate dataset
data_size = 500

generator = ArabicConversationalGenerator(seed=SEED)
samples = generator.generate_samples(num_samples=data_size)

# Get statistics
stats = generator.get_statistics()
print(f"\nDataset Statistics:")
print(f"  Total samples: {stats['total_samples']}")
print(f"  Unique inputs: {stats['unique_inputs']}")
print(f"  Unique outputs: {stats['unique_outputs']}")
print(f"  Avg input length: {stats['avg_input_length']:.1f} chars")
print(f"  Avg output length: {stats['avg_output_length']:.1f} chars")

In [ ]:
# Split into train/val/test
train_samples, val_samples, test_samples = generator.split_data(
    train_ratio=0.8,
    val_ratio=0.1,
    test_ratio=0.1
)

print(f"Train: {len(train_samples)} samples")
print(f"Val:   {len(val_samples)} samples")
print(f"Test:  {len(test_samples)} samples")

## 3. Data Exploration <a id='3-explore'></a>

In [ ]:
# Type distribution
type_counts = stats['type_distribution']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Conversation type distribution
ax1 = axes[0]
types = list(type_counts.keys())
counts = list(type_counts.values())
colors = plt.cm.Set3(np.linspace(0, 1, len(types)))
bars = ax1.bar(types, counts, color=colors)
ax1.set_xlabel('Conversation Type')
ax1.set_ylabel('Count')
ax1.set_title('Distribution by Conversation Type')
ax1.tick_params(axis='x', rotation=45)
for bar, count in zip(bars, counts):
    ax1.annotate(f'{count}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                 xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)

# Language distribution
ax2 = axes[1]
lang_counts = stats['language_distribution']
ax2.pie(lang_counts.values(), labels=lang_counts.keys(), autopct='%1.1f%%',
        colors=['#66b3ff', '#99ff99'])
ax2.set_title('Language Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Show sample conversations by type
print("Sample Conversations by Type:")
print("=" * 70)

for conv_type in type_counts.keys():
    type_samples = [s for s in samples if s.conversation_type == conv_type][:2]
    print(f"\n{conv_type.upper()}:")
    for i, sample in enumerate(type_samples):
        print(f"  Q: {sample.input_text}")
        print(f"  A: {sample.output_text}")
        if i < len(type_samples) - 1:
            print()

## 4. Tokenization <a id='4-tokenize'></a>

In [ ]:
# Create tokenizer
tokenizer_config = TokenizerConfig(
    max_seq_len=32,
    min_freq=1
)

tokenizer = create_tokenizer_from_samples(train_samples, max_seq_len=32, min_freq=1)

# Get vocabulary summary
summary = tokenizer.get_summary()
print("Tokenizer Summary:")
for key, value in summary.items():
    print(f"  {key}: {value}")

In [ ]:
# Encode data
train_inputs = [s.input_text for s in train_samples]
train_outputs = [s.output_text for s in train_samples]
val_inputs = [s.input_text for s in val_samples]
val_outputs = [s.output_text for s in val_samples]
test_inputs = [s.input_text for s in test_samples]
test_outputs = [s.output_text for s in test_samples]

# Encode inputs
X_train = tokenizer.batch_encode_inputs(train_inputs)
X_val = tokenizer.batch_encode_inputs(val_inputs)
X_test = tokenizer.batch_encode_inputs(test_inputs)

# Build output label mapping from training data FIRST
# This ensures consistent class IDs across train/val/test
tokenizer.build_output_label_mapping(train_outputs)

# Now encode outputs as class labels (using the mapping built from train)
y_train = tokenizer.batch_encode_output_labels(train_outputs)
y_val = tokenizer.batch_encode_output_labels(val_outputs)
y_test = tokenizer.batch_encode_output_labels(test_outputs)

# Check for unseen labels in val/test (will be mapped to 0)
val_unseen = sum(1 for o in val_outputs if o not in tokenizer._response_to_id)
test_unseen = sum(1 for o in test_outputs if o not in tokenizer._response_to_id)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"Number of classes: {tokenizer.num_classes}")
print(f"Max label in y_train: {y_train.max()}")
print(f"Max label in y_val: {y_val.max()}")
print(f"Max label in y_test: {y_test.max()}")
print(f"Unseen responses in val: {val_unseen} ({val_unseen/len(val_outputs)*100:.1f}%)")
print(f"Unseen responses in test: {test_unseen} ({test_unseen/len(test_outputs)*100:.1f}%)")

## 5. Standard Transformer <a id='5-standard'></a>

Train a standard transformer with fixed architecture as baseline.

In [ ]:
# Configure standard transformer
std_config = TransformerConfig(
    vocab_size=tokenizer.input_vocab_size,
    max_seq_len=32,
    embed_dim=64,
    num_heads=4,
    num_layers=3,
    ffn_dim=256,
    dropout=0.1,
    num_classes=tokenizer.num_classes,
    seed=SEED
)

print("Standard Transformer Configuration:")
print(f"  Vocab size: {std_config.vocab_size}")
print(f"  Embed dim: {std_config.embed_dim}")
print(f"  Num heads: {std_config.num_heads}")
print(f"  Num layers: {std_config.num_layers}")
print(f"  FFN dim: {std_config.ffn_dim}")
print(f"  Num classes: {std_config.num_classes}")

In [ ]:
# Create and train standard transformer
standard_model = StandardTransformer(std_config)
print(f"Total parameters: {standard_model.count_parameters():,}")

# Train
std_history = standard_model.fit(
    X_train, y_train,
    X_val=X_val, y_val=y_val,
    epochs=50,
    batch_size=32,
    learning_rate=0.005,
    verbose=True
)

In [ ]:
# Evaluate standard transformer
std_eval = standard_model.evaluate(X_test, y_test)
print("\nStandard Transformer - Test Results:")
print(f"  Accuracy: {std_eval['accuracy']:.2%}")
print(f"  Top-3 Accuracy: {std_eval['top3_accuracy']:.2%}")
print(f"  Top-5 Accuracy: {std_eval['top5_accuracy']:.2%}")
print(f"  Perplexity: {std_eval['perplexity']:.2f}")

## 6. Dynamic Transformer <a id='6-dynamic'></a>

Train a Dynamic NN-enhanced transformer with adaptive architecture:
- Dynamic attention heads
- Dynamic encoder layers
- Dynamic FFN width
- 3-phase training (Exploration, Estimation, Main)

In [ ]:
# Configure dynamic transformer
dyn_config = DynamicTransformerConfig(
    vocab_size=tokenizer.input_vocab_size,
    max_seq_len=32,
    embed_dim=64,
    initial_num_heads=4,
    initial_num_layers=3,
    initial_ffn_dim=256,
    min_heads=2,
    max_heads=80,
    min_layers=1,
    max_layers=60,
    min_ffn_dim=86,
    max_ffn_dim=512,
    dropout=0.1,
    num_classes=tokenizer.num_classes,
    seed=1
)

print("Dynamic Transformer Configuration:")
print(f"  Initial layers: {dyn_config.initial_num_layers} (range: {dyn_config.min_layers}-{dyn_config.max_layers})")
print(f"  Initial heads: {dyn_config.initial_num_heads} (range: {dyn_config.min_heads}-{dyn_config.max_heads})")
print(f"  Initial FFN: {dyn_config.initial_ffn_dim} (range: {dyn_config.min_ffn_dim}-{dyn_config.max_ffn_dim})")

In [ ]:
# Create and train dynamic transformer
import importlib
from src import transformer_components, dynamic_transformer
importlib.reload(transformer_components)
importlib.reload(dynamic_transformer)

dynamic_model = DynamicTransformer(dyn_config)
print(f"Initial parameters: {dynamic_model.count_parameters():,}")

# Train with 3-phase approach
dyn_history = dynamic_model.fit(
    X_train, y_train,
    X_val=X_val, y_val=y_val,
    batch_size=32,
    verbose=True
)

In [ ]:
# Evaluate dynamic transformer
dyn_eval = dynamic_model.evaluate(X_test, y_test)
print("\nDynamic Transformer - Test Results:")
print(f"  Accuracy: {dyn_eval['accuracy']:.2%}")
print(f"  Top-3 Accuracy: {dyn_eval['top3_accuracy']:.2%}")
print(f"  Top-5 Accuracy: {dyn_eval['top5_accuracy']:.2%}")
print(f"  Perplexity: {dyn_eval['perplexity']:.2f}")

# Health status
health = dynamic_model.health_status()
print(f"\nHealth Status: {health['state']}")
print(f"  Cancer Score: {health['cancer_score']:.2%}")
print(f"  Alzheimer Score: {health['alzheimer_score']:.2%}")

## 7. Side-by-Side Comparison <a id='7-comparison'></a>

In [ ]:
# Create comparison evaluator
evaluator = TransformerComparisonEvaluator(
    standard_model=standard_model,
    dynamic_model=dynamic_model,
    tokenizer=tokenizer
)

# Run full comparison
metrics = evaluator.evaluate(X_test, y_test, verbose=True)

In [ ]:
# Plot training curves
evaluator.plot_training_curves()

In [ ]:
# Parameter efficiency comparison
evaluator.plot_parameter_efficiency()

## 8. Architecture Evolution Analysis <a id='8-architecture'></a>

Visualize how the Dynamic Transformer's architecture evolved during training.

In [ ]:
# Plot architecture evolution
evaluator.plot_architecture_evolution()

In [ ]:
# Final architecture summary
print("\nFinal Dynamic Architecture:")
arch = dynamic_model.get_architecture_summary()
print(f"  Layers: {arch['num_layers']}")
print(f"  Heads per layer: {arch['heads_per_layer']}")
print(f"  FFN dim per layer: {arch['ffn_dim_per_layer']}")
print(f"  Total parameters: {arch['total_parameters']:,}")

## 9. Health Monitoring Analysis <a id='9-health'></a>

Analyze the health scores (Cancer/Alzheimer) that track architecture stability.

In [ ]:
# Plot health scores
evaluator.plot_health_scores()

In [ ]:
# Architecture change summary
print("\nArchitecture Changes Summary:")
print(f"  Layers: +{dyn_history['layers_added']} / -{dyn_history['layers_removed']}")
print(f"  Attention Heads: +{dyn_history['heads_added']} / -{dyn_history['heads_removed']}")
print(f"  FFN Nodes: +{dyn_history['nodes_added']} / -{dyn_history['nodes_removed']}")
print(f"  Perturbations Applied: {dyn_history['perturbations_applied']}")

## 10. Prediction Demo <a id='10-demo'></a>

Interactive demonstration of both models on sample inputs.

In [ ]:
# Show prediction examples
evaluator.show_prediction_examples(X_test, y_test, n_examples=10, show_incorrect=True)

In [ ]:
# Interactive test
def test_models(input_text):
    """Test both models on a custom input"""
    # Encode input
    input_ids = tokenizer.encode_input(input_text)
    input_batch = input_ids.reshape(1, -1)
    
    # Standard prediction
    std_probs = standard_model.predict_proba(input_batch)[0]
    std_pred = np.argmax(std_probs)
    std_conf = std_probs[std_pred]
    
    # Dynamic prediction
    dyn_probs = dynamic_model.predict_proba(input_batch)[0]
    dyn_pred = np.argmax(dyn_probs)
    dyn_conf = dyn_probs[dyn_pred]
    
    print(f"Input: {input_text}")
    print(f"Standard: {tokenizer.decode_output_label(std_pred)} ({std_conf:.2%})")
    print(f"Dynamic:  {tokenizer.decode_output_label(dyn_pred)} ({dyn_conf:.2%})")
    print()

# Test with some Arabic phrases
test_inputs = [
    "السلام عليكم",
    "ما اسمك",
    "كم الساعة",
    "كيف الطقس",
    "Hello كيف حالك"
]

print("\nInteractive Demo:")
print("=" * 50)
for text in test_inputs:
    test_models(text)

## 11. Conclusions <a id='11-conclusions'></a>

In [ ]:
# Generate and print comparison report
report = evaluator.generate_comparison_report()
print(report)

In [ ]:
# Final summary
print("\n" + "=" * 70)
print("FINAL SUMMARY")
print("=" * 70)

acc_diff = metrics.dynamic_accuracy - metrics.standard_accuracy
param_diff = metrics.dynamic_final_params - metrics.standard_params

print(f"\nAccuracy: Dynamic {'outperforms' if acc_diff > 0 else 'underperforms'} Standard by {abs(acc_diff):.2%}")
print(f"Parameters: Dynamic uses {abs(param_diff):,} {'more' if param_diff > 0 else 'fewer'} parameters")

if acc_diff > 0 and param_diff < 0:
    print("\nConclusion: Dynamic NN IMPROVES transformer - better accuracy with fewer parameters!")
elif acc_diff > 0:
    print("\nConclusion: Dynamic NN achieves better accuracy but uses more parameters.")
elif param_diff < 0:
    print("\nConclusion: Dynamic NN uses fewer parameters but with lower accuracy.")
else:
    print("\nConclusion: Standard transformer performs better on this task.")

print("\n" + "=" * 70)

In [ ]:
# Save models
import os
models_dir = '../models'
os.makedirs(models_dir, exist_ok=True)

standard_model.save(f'{models_dir}/standard_transformer.npz')
dynamic_model.save(f'{models_dir}/dynamic_transformer.npz')
tokenizer.save(f'{models_dir}/tokenizer.json')

print(f"Models saved to {models_dir}/")